# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [4]:
from dotenv import load_dotenv
# TODO: Load environment variables
load_dotenv()

openai_key = os.getenv('OPENAI_API_KEY')
tavily_key = os.getenv('TAVILY_API_KEY')

print(openai_key)  # This will print your OpenAI API key
print(tavily_key)  # This will print your Tavily API key

voc-128284763016886546721826946fbfde31796.64664478
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


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [7]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

from lib.tooling import tool
import chromadb

# Initialize ChromaDB client and collection
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most relevant results in the vector DB.
    Args:
        query (str): a question about the game industry.
    Returns:
        List[dict]: Each element contains Platform, Name, YearOfRelease, Description.
    """
    # Perform semantic search
    results = collection.query(
        query_texts=[query],
        n_results=15  # Number of results to return
    )
    games = []
    for metadata in results['metadatas'][0]:
        games.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description")
        })
    return games

In [9]:
results = retrieve_game("give info on Pokémon Gold and Silver")
print(results)

[{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'GameCube', 'Name': 'Super Smash Bros. Melee', 'YearOfRelease': 2001, 'Description': 'A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.'}, {'Platform': 'PlayStation 3', 'Name': 'Gran Turismo 5', 'YearOfRelease': 2010, 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.'}, {'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Description': "A groundbreaking 3D platformer that set new standards for the genre

#### Evaluate Retrieval Tool

In [12]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

import ollama
import json


@tool
def evaluate_retrieval_tool(question, document):
# prompts can be improved further
    prompt = f"""
You are evaluating whether a retrieved document is useful to answer a user's question.
Consider developer and publisher as same thing.

Question:
{question}

Retrieved document:
{document}

Respond ONLY in JSON format:

{{
 "useful": true or false,
 "reason": "give reason if loaded document is relevant"
}}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}]
    )

    content = response["message"]["content"]

    try:
        return json.loads(content)
    except:
        return {"useful": False, "reason": "Could not parse response"}



In [13]:
# Example question
question = "when was Pokémon Gold and Silver released?"

# Example document (use one from retrieve_game)
retrieved = retrieve_game(question)
document = retrieved[0]  # Use the first result

# Test the tool
result = evaluate_retrieval_tool(question, document)
print(result)

{'useful': True, 'reason': "The document provides the 'YearOfRelease' for the game 'Pokémon Gold and Silver', which matches the user's question."}


#### Game Web Search Tool

In [14]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search_tool(query):
    """
    Tool: Perform web search when internal knowledge is insufficient
    """

    results = tavily.search(
        query=query,
        max_results=3
    )

    return results

In [11]:
web_search_tool("When was counter strike released?")

{'query': 'When was counter strike released?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.quora.com/When-did-Counter-Strike-come-out',
   'title': 'When did Counter-Strike come out?',
   'content': 'Counter Strike: Source (the full game) was release on November 1, 2004. The beta, however, was released on August 11, 2004, but only to members',
   'score': 0.99939764,
   'raw_content': None},
  {'url': 'https://tradeit.gg/blog/counter-strike-release-date-for-every-game-in-the-series/?srsltid=AfmBOopCjNAam092AyHQDEOh77OMNfLWp2PK0F4AIeYz_swrDTJEacU6',
   'title': 'Counter-Strike Release Date for Every Game in the Series',
   'content': '# Counter-Strike Release Date for Every Game in the Series. We’re about to take a tactical breach into the release dates of every game in the Counter-Strike series. * Counter-Strike started as a Half-Life mod and quickly evolved into a hit series with several notable games including the latest Counter-Str

### Agent

In [15]:
def generate_answer_response(question: str, data: any, source: str) -> str:
    """Generate answer from data"""
    if source == "internal database":
        games = data['games']
        answer = f"Based on our internal database:\n\n"
        for game in games[:2]:
            answer += f"- {game['Name']} was released in {game['YearOfRelease']} for {game['Platform']}\n"
            #answer += f"  Genre: {game['Genre']}, Publisher: {game['Publisher']}\n"
            answer += f"  {game['Description']}\n\n"
        answer += "Source: Internal Game Database"
    else:
        answer = f"Based on web search:\n\n"
        for item in data[:2]:
            answer += f"- {item['title']}\n"
            answer += f"  {item['content'][:200]}...\n"
            answer += f"  Source: {item['url']}\n\n"
    
    return answer

In [ ]:
def rewrite_query_with_context(current_question: str, history: list) -> str:
    """
    Rewrites a follow-up question into a standalone query using conversation history.
    Uses Ollama (Mistral).
    """

    if not history:
        return current_question

    formatted_history = ""
    for turn in history[-2:]:
        formatted_history += f"User: {turn['question']}\nAI: {turn['answer']}\n"

    prompt = f"""
Given the following conversation history and the user's new question, 
rewrite the new question to be a standalone query that contains all necessary context.

If the new question is independent, return it unchanged.

DO NOT answer the question. ONLY rewrite it.

Conversation History:
{formatted_history}

New Question: {current_question}

Standalone Question:
"""

    try:
        response = ollama.chat(
            model="mistral",
            messages=[{"role": "user", "content": prompt}]
        )

        standalone_query = response["message"]["content"].strip()
        print(f"🧠 [Memory] Rewrote query to: {standalone_query}")

        return standalone_query

    except Exception as e:
        print(f"⚠️ Rewrite failed: {e}")
        return current_question

In [18]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

class UdaPlayAgent:
    """Agent with explicit state machine workflow"""
    
    def __init__(self):
        self.conversation_history = {}
    
    def query(self, question: str, session_id: str = "default") -> dict:
        current_history = self.conversation_history.get(session_id, [])

        standalone_question = rewrite_query_with_context(
            question,
            current_history
        )
        """Process query through explicit state machine"""
        
        state = "RETRIEVE"
        retrieval_result = None
        evaluation = None
        web_results = None
        final_answer = None
        citations = []
        
        while state != "END":
            
            if state == "RETRIEVE":
                retrieval_result = retrieve_game(standalone_question)
                state = "EVALUATE"
            
            elif state == "EVALUATE":
                evaluation = evaluate_retrieval_tool(standalone_question, retrieval_result)
                if evaluation["useful"]:
                    state = "ANSWER"
                else:
                    state = "WEB"
            
            elif state == "WEB":
                web_results = web_search_tool(standalone_question)
        
                if not web_results:
                    citations = "No web results found"
                else:
                    results = web_results.get("results", [])
                    if not results:
                        citations = "No web results found"
                    else:
                        first_result = results[0] if isinstance(results, list) else {}
                        citations = first_result.get("url", "No URL available")
                state = "ANSWER"
            
            elif state == "ANSWER":
                if evaluation["useful"]:
                    final_answer = generate_answer_response(standalone_question, {"games": retrieval_result}, "internal database")  # Wrap in dict
                else:
                    final_answer = generate_answer_response(standalone_question, web_results['results'], "web search")  # Access 'results' key
                state = "END"
        
        result = {
            "question": question,
            "retrieval_used": True,
            "evaluation": evaluation,
            "web_fallback_triggered": not evaluation["useful"],
            "sources": citations if citations else "Internal Database",
            "answer": final_answer
        }
        self.conversation_history.setdefault(session_id, []).append(result)
        return result
    
    def print_report(self, result: dict):
        """Print structured report"""
        print("\n" + "="*80)
        print("QUERY REPORT")
        print("="*80)
        print(f"Question: {result['question']}")
        print()
        print("Tool Usage:")
        print("  ✓ retrieve_game() - Called")
        print("  ✓ evaluate_retrieval() - Called")
        if result['web_fallback_triggered']:
            print("  ✓ game_web_search() - Called")
        print()
        print("Evaluation Result:")
        print(f"  Useful: {result['evaluation']['useful']}")
        print(f"  Reason: {result['evaluation']['reason']}")
        print()
        print(f"Web Fallback Triggered: {result['web_fallback_triggered']}")
        print("Below details can help with the query")
        print("Sources:")
        if isinstance(result['sources'], list):
            for url in result['sources']:
                print(f"  - {url}")
        else:
            print(f"  - {result['sources']}")
        print()
        print("="*80)
        print("FINAL ANSWER")
        print("="*80)
        print(result['answer'])
        print()
    
    def print_conversation_history(self, session_id: str = None):
        """Print formatted conversation history"""
        if session_id:
            history = self.conversation_history.get(session_id, [])
            if not history:
                print(f"No conversation history available for session '{session_id}'.")
                return
        else:
            history = []
            for sess_hist in self.conversation_history.values():
                history.extend(sess_hist)
            if not history:
                print("No conversation history available.")
                return
        
        print("\n" + "="*100)
        print("CONVERSATION HISTORY")
        if session_id:
            print(f"Session: {session_id}")
        print("="*100)
        print(f"Total conversations: {len(history)}")
        print()
        
        for i, result in enumerate(history, 1):
            print(f"--- Conversation {i} ---")
            print(f"Question: {result['question']}")
            print(f"Retrieval Used: {result['retrieval_used']}")
            print(f"Web Fallback Triggered: {result['web_fallback_triggered']}")
            print(f"Evaluation Useful: {result['evaluation']['useful']}")
            print(f"Answer Preview: {result['answer'][:100]}...")
            print()
        
        print("="*100)
    
    def get_conversation_history(self, session_id: str = None):
        if session_id:
            return self.conversation_history.get(session_id, [])
        else:
            history = []
            for sess_hist in self.conversation_history.values():
                history.extend(sess_hist)
            return history

agent = UdaPlayAgent()
print("✓ UdaPlay Agent ready")

✓ UdaPlay Agent ready


In [21]:
result1 = agent.query("tell me something about spider man 2 game?", session_id="udaplay_demo")
agent.print_report(result1)


🧠 [Memory] Rewritten: What is known about the Spider-Man 2 video game? (Or) Can you provide information on the Spider-Man 2 game? (If the user is specifically asking about the video game)

QUERY REPORT
Question: tell me something about spider man 2 game?

Tool Usage:
  ✓ retrieve_game() - Called
  ✓ evaluate_retrieval() - Called

Evaluation Result:
  Useful: True
  Reason: The retrieved document includes information about Marvel's Spider-Man 2 (for PlayStation 5), which is a sequel to the acclaimed Spider-Man game and features both Peter Parker and Miles Morales as playable characters. This aligns with the user's question asking for information on the Spider-Man 2 game.

Web Fallback Triggered: False
Below details can help with the query
Sources:
  - Internal Database

FINAL ANSWER
Based on our internal database:

- Marvel's Spider-Man 2 was released in 2023 for PlayStation 5
  The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable charac

In [22]:
result1 = agent.query("when was it released?", session_id="udaplay_demo")
agent.print_report(result1)

🧠 [Memory] Rewritten: When was the Spider-Man 2 game released? (Specify the year and platform if possible.)

QUERY REPORT
Question: when was it released?

Tool Usage:
  ✓ retrieve_game() - Called
  ✓ evaluate_retrieval() - Called

Evaluation Result:
  Useful: True
  Reason: The retrieved document contains the information for 'Marvel's Spider-Man 2', which is the game being asked about. The year of release is provided as 2023.

Web Fallback Triggered: False
Below details can help with the query
Sources:
  - Internal Database

FINAL ANSWER
Based on our internal database:

- Marvel's Spider-Man 2 was released in 2023 for PlayStation 5
  The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.

- Marvel's Spider-Man was released in 2018 for PlayStation 4
  An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.

Source: Internal Game Database



In [23]:
result3 = agent.query("Tell me about mortal kombat game?", session_id="udaplay_demo")
agent.print_report(result3)

🧠 [Memory] Rewritten: What is the information available in our internal database regarding Mortal Kombat games?

QUERY REPORT
Question: Tell me about mortal kombat game?

Tool Usage:
  ✓ retrieve_game() - Called
  ✓ evaluate_retrieval() - Called
  ✓ game_web_search() - Called

Evaluation Result:
  Useful: False
  Reason: The retrieved document does not contain any information about Mortal Kombat games.

Web Fallback Triggered: True
Below details can help with the query
Sources:
  - https://gamesdb.launchbox-app.com/games/details/2772-mortal-kombat

FINAL ANSWER
Based on web search:

- Mortal Kombat - LaunchBox Games Database
  In Mortal Kombat, the player receives information concerning the backstories of the characters and their relationships with one another...
  Source: https://gamesdb.launchbox-app.com/games/details/2772-mortal-kombat

- Series: Mortal Kombat - IGDB.com
  All games in the Mortal Kombat Series (79). Filters. Select an option, Amiga, Android, Arcade, DOS, DC, Game Bo

In [24]:
result3 = agent.query("when was this released first time?", session_id="udaplay_demo")
agent.print_report(result3)

🧠 [Memory] Rewritten: What is the release year of the first Mortal Kombat game?

QUERY REPORT
Question: when was this released first time?

Tool Usage:
  ✓ retrieve_game() - Called
  ✓ evaluate_retrieval() - Called
  ✓ game_web_search() - Called

Evaluation Result:
  Useful: False
  Reason: The retrieved document does not contain information about the first Mortal Kombat game. It only lists various games from different platforms and genres.

Web Fallback Triggered: True
Below details can help with the query
Sources:
  - https://en.wikipedia.org/wiki/Mortal_Kombat

FINAL ANSWER
Based on web search:

- Mortal Kombat - Wikipedia
  # *Mortal Kombat*. | First release | *Mortal Kombat "Mortal Kombat (1992 video game)")* August 1992 |. ***Mortal Kombat*** is an American media franchise centered on a series of fighting video games o...
  Source: https://en.wikipedia.org/wiki/Mortal_Kombat

- Mortal Kombat (1992 video game)
  Mortal Kombat is the first game in the Mortal Kombat fighting game se

In [25]:
result3 = agent.query("when was brian lara cricket 99 released?", session_id="session2")
agent.print_report(result3)


QUERY REPORT
Question: when was brian lara cricket 99 released?

Tool Usage:
  ✓ retrieve_game() - Called
  ✓ evaluate_retrieval() - Called
  ✓ game_web_search() - Called

Evaluation Result:
  Useful: False
  Reason: The retrieved document does not contain information about the game 'Brian Lara Cricket 99'. The records pertain to different games, released on various platforms and genres, with no match for the requested title.

Web Fallback Triggered: True
Below details can help with the query
Sources:
  - https://giantbomb.com/wiki/Games/Brian_Lara_Cricket_99

FINAL ANSWER
Based on web search:

- Brian Lara Cricket 99 (Game) - Giant Bomb Video Game Wiki
  Brian Lara Cricket '99 (BLC99) was released in 1998 to a worldwide audience. Though to players in Australia and New Zealand it was officially endorsed by...
  Source: https://giantbomb.com/wiki/Games/Brian_Lara_Cricket_99

- Brian Lara Cricket '99 Release Information for PC - GameFAQs
  Brian Lara Cricket '99 is a Sports game, develo

In [26]:
result3 = agent.query("how were the reviews of this game?", session_id="session2")
agent.print_report(result3)

🧠 [Memory] Rewritten: What was the critical reception or reviews for Brian Lara Cricket 99?

QUERY REPORT
Question: how were the reviews of this game?

Tool Usage:
  ✓ retrieve_game() - Called
  ✓ evaluate_retrieval() - Called
  ✓ game_web_search() - Called

Evaluation Result:
  Useful: False
  Reason: The retrieved document does not contain information about the critical reception of Brian Lara Cricket 99. It only lists various video games across different platforms and years.

Web Fallback Triggered: True
Below details can help with the query
Sources:
  - https://www.gamespot.com/brian-lara-cricket/user-reviews/2200-406563/

FINAL ANSWER
Based on web search:

- king_pickman's Review of Brian Lara Cricket (Value Series)
  After all these years it's still the best cricket game to date · Just another part of cricket, without ghraphics or gameplay. Review Score: 6.7....
  Source: https://www.gamespot.com/brian-lara-cricket/user-reviews/2200-406563/

- Brian Lara Cricket Review for PlaySt

In [27]:
agent.print_conversation_history(session_id="udaplay_demo")

# Access raw data
history = agent.get_conversation_history(session_id="udaplay_demo")
print(f"Total conversations: {len(history)}")


CONVERSATION HISTORY
Session: udaplay_demo
Total conversations: 6

--- Conversation 1 ---
Question: tell me something about spider man 2?
Retrieval Used: True
Web Fallback Triggered: True
Evaluation Useful: False
Answer Preview: Based on web search:

- 'Spider-Man 2': Everything you need to know about the blockbuster ...
  Spid...

--- Conversation 2 ---
Question: when was it released?
Retrieval Used: True
Web Fallback Triggered: True
Evaluation Useful: False
Answer Preview: Based on web search:

- Spider-Man 2 (released 20 years ago today on June 30th, 2004 ...
  20 years ...

--- Conversation 3 ---
Question: tell me something about spider man 2 game?
Retrieval Used: True
Web Fallback Triggered: False
Evaluation Useful: True
Answer Preview: Based on our internal database:

- Marvel's Spider-Man 2 was released in 2023 for PlayStation 5
  Th...

--- Conversation 4 ---
Question: when was it released?
Retrieval Used: True
Web Fallback Triggered: False
Evaluation Useful: True
Answer Previ

### (Optional) Advanced

In [28]:
agent.print_conversation_history(session_id="session2")

# Access raw data
history = agent.get_conversation_history(session_id="session2")
print(f"Total conversations: {len(history)}")


CONVERSATION HISTORY
Session: session2
Total conversations: 2

--- Conversation 1 ---
Question: when was brian lara cricket 99 released?
Retrieval Used: True
Web Fallback Triggered: True
Evaluation Useful: False
Answer Preview: Based on web search:

- Brian Lara Cricket 99 (Game) - Giant Bomb Video Game Wiki
  Brian Lara Crick...

--- Conversation 2 ---
Question: how were the reviews of this game?
Retrieval Used: True
Web Fallback Triggered: True
Evaluation Useful: False
Answer Preview: Based on web search:

- king_pickman's Review of Brian Lara Cricket (Value Series)
  After all these...

Total conversations: 2


In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes